In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [3]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

12.62

In [4]:
def calculate_heikin_ashi(df):
    ha_close = (df['open'] + df['high'] + df['low'] + df['close']) / 4
    ha_open = (ha_close.shift(1) + ha_close.shift(1)) / 2
    ha_high = df[['high', 'open', 'close']].max(axis=1)
    ha_low = df[['low', 'open', 'close']].min(axis=1)

    return pd.DataFrame({'ha_open': ha_open, 'ha_high': ha_high, 'ha_low': ha_low, 'ha_close': ha_close, 'time':df.time})

In [ ]:
import numpy as np
import pandas as pd
import pandas_ta as pdt

def supertrend(factor, atr_length, high, low, close):
    atr = pdt.atr(high, low, close, atr_length)
    basic_upper_band = (high + low) / 2 + factor * atr
    basic_lower_band = (high + low) / 2 - factor * atr
    bullish_signal = close > basic_upper_band
    bearish_signal = close < basic_lower_band
    bullish_supertrend = np.full_like(close, np.nan)
    bearish_supertrend = np.full_like(close, np.nan)

    for i in range(1, len(close)):
        if bullish_signal[i] or (bullish_supertrend[i-1] and close[i-1] > basic_upper_band[i-1]):
            bullish_supertrend[i] = max(basic_upper_band[i], bullish_supertrend[i-1])
        else:
            bullish_supertrend[i] = basic_upper_band[i]

        if bearish_signal[i] or (bearish_supertrend[i-1] and close[i-1] < basic_lower_band[i-1]):
            bearish_supertrend[i] = min(basic_lower_band[i], bearish_supertrend[i-1])
        else:
            bearish_supertrend[i] = basic_lower_band[i]

    direction = np.where(close > bullish_supertrend, 1, np.where(close < bearish_supertrend, -1, 0))
    supertrend = np.where(direction == 1, bullish_supertrend, bearish_supertrend)
    
    return supertrend, direction

# Example usage:
# Assuming df is your DataFrame containing OHLC data
# Replace this with your actual DataFrame
# Example:
# df = pd.DataFrame({'open': [...], 'high': [...], 'low': [...], 'close': [...]})

# Convert input parameters from Pine Script to Python
# factor = 3.0
# atr_length = 10




In [ ]:
pdt.atr?

In [4]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertrend
    factor = 3.0
    atr_length = 10
    supertrend_values, direction = supertrend(factor, atr_length, rates_frame['high'], rates_frame['low'], rates_frame['close'])
    rates_frame['spvalues'] = supertrend_values
    rates_frame['direction'] = direction
    
    # Print or access the supertrend_values and direction arrays
    print("Supertrend values:", supertrend_values)
    print("Direction:", direction)
    return rates_frame

In [5]:
def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 12)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertren
    return rates_frame

In [ ]:
a = get_values('GBPUSD', 1000, 50, 'D1')

In [ ]:
a

In [ ]:
type(direction)

In [5]:
def ema(s, n):
    ema = []
    zero = [0]*(20000-19801)
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)
    am = zero + ema
    print(len(am))
    return am

In [6]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [7]:
def calculate_rsi(data, window):
    # Calculate the differences in the data
    delta = data.diff()

    # Make the positive gains (up) and negative gains (down) Series
    gain = (delta.where(delta > 0, 0)).fillna(0)
    loss = (-delta.where(delta < 0, 0)).fillna(0)

    # Calculate the average gain and average loss
    avg_gain = gain.rolling(window=window, min_periods=1).mean()
    avg_loss = loss.rolling(window=window, min_periods=1).mean()

    # Calculate the RS (Relative Strength)
    rs = avg_gain / avg_loss

    # Calculate the RSI
    rsi = 100 - (100 / (1 + rs))

    return rsi


In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
# GBPUSD = 0.00100
# EURUSD = 0.00050
# USDJPY = 0.491
# a = get_values('USDJPY', 10000, 100,12, 'H1')
# a = get_values(symbol, 10000, 100,21, 'H4')
# a = get_values(symbol, 10000, 50,21, 'D1')
# a = get_values(symbol, 10000, 50, 12, 'H1')
a = get_values(symbol, 10000, 50,9, 'M30')

lot = 0.1
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

# 
print("here")
print(lot)
for i in range(1, len(a)):
    if check==0:
#         if a.iloc[i-1].close <= a.iloc[i-1].sma and  a.iloc[i-1].open >= a.iloc[i-1].sma:
#             print(f"{a.iloc[i].name}")
#             buy_price = a.iloc[i].open
#             check=1
            
        if a.iloc[i].rsi < 25 and  a.iloc[i-1].rsi >= 50 and a.iloc[i]:
            print(f"{a.iloc[i].name}--- {a.iloc[i].open} --{a.iloc[i].rsi}--{a.iloc[i].close}")
            buy_price = a.iloc[i].open
            diff = abs(a.iloc[i].close - a.iloc[i].open)/2
            check=2
#             continue
#     if check==1:
#         sell_price = a.iloc[i].close
#         pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
# #         print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#         if a.iloc[i].high >= a.iloc[i].sma and  (a.iloc[i].high - a.iloc[i].sma) >= 0.00300:
#             pp1 = price_action(symbol, lot, buy_price, a.iloc[i].sma+0.00300, mt5.ORDER_TYPE_SELL)
#             print(f"PP1 ---> {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}-{a.iloc[i].name}")
#             if pp1<=-10:
#                 profit.append(-10)
#             else:
#                 profit.append(pp1)
#             check=0
#         elif pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
#             check = 0
            
    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#         profit.append(pp)
#         check=0
#         if a.iloc[i].low <= a.iloc[i].sma and  (a.iloc[i].sma - a.iloc[i].low) >= 0.00300:
#             pp1 = price_action(symbol, lot, buy_price, a.iloc[i].sma-0.00300, mt5.ORDER_TYPE_BUY)
#             print(f"PP1 ---> {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}-{a.iloc[i].name}")
#             if pp1<=-20:
#                 profit.append(-20)
#             else:
#                 profit.append(pp1)
#             check=0
#         elif pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if a.iloc[i].rsi < a.iloc[i-1].rsi or pp < 0.0:
            print(f"pp- {pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#             print(f"Diff--> {diff}")
#             if pp <0.0 and sell_price < (a.iloc[i].open-diff):
#                 pp1 = price_action(symbol, lot, buy_price, a.iloc[i].close-diff, mt5.ORDER_TYPE_BUY)
#                 print(f"pp1- {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                 profit.append(pp1)
#                 check = 0
#             elif sell_price < (a.iloc[i].open-diff):
# #             else:
#                 print(f"pp- {pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                 profit.append(pp)
#                 check = 0

In [129]:
def get_spread_amount(symbol, lot_size=1.0):
    # Get symbol information
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        print(f"Failed to get symbol info for {symbol}")
        return None
    
    # Get tick data for the symbol
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Failed to get tick data for {symbol}")
        return None

    # Calculate the spread in points (pips)
    spread = tick.ask - tick.bid
    points = spread / symbol_info.point

    # Calculate the spread amount in USD
    spread_amount = points * symbol_info.point * lot_size * symbol_info.trade_tick_value
    return spread_amount


In [83]:
def get_values(symbol, size, smaa=50,r=7, t='M30'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H6':mt5.TIMEFRAME_H6,'H8':mt5.TIMEFRAME_H8, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['rsi'] = get_rsi(rates_frame['close'], r)
#     rates_frame['rsi'] = calculate_rsi(rates_frame['close'], r)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()

    rates_frame['sma1'] = rates_frame['close'].rolling(window=9).mean()
    rates_frame['sma2'] = rates_frame['close'].rolling(window=21).mean()
    rates_frame = rates_frame[rates_frame['sma1'].notna()]
    rates_frame = rates_frame[rates_frame['sma2'].notna()]

    # Calculate Supertren
    return rates_frame

In [169]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = 0
# GBPUSD = 0.00100
# EURUSD = 0.00050
# USDJPY = 0.491
# a = get_values('USDJPY', 10000, 100,12, 'H1')
# a = get_values(symbol, 10000, 100,21, 'H4')
# a = get_values(symbol, 10000, 50,21, 'D1')
# a = get_values(symbol, 10000, 50, 12, 'H1')

#BTCUSD
# a = get_values(symbol, 10000, 50,21, 'M30')
#GBPJPY
a = get_values(symbol, 10000, 100,7, 'M15')
# a = get_values('GBPJPY', 10000, 50,21, 'H2')
# a = get_values('GBPJPY', 10000, 50,21, 'H4')
# a = get_values('GBPJPY', 10000, 50,21, 'H6')


lot = 0.1
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)
spread = get_spread_amount(symbol, lot)
# 
print(spread)
print(lot)
for i in range(1, len(a)-1):
    if check==0:
#         if a.iloc[i].rsi< 30.0 and  a.iloc[i].rsi >36.0:
#             print(f"{a.iloc[i].name}--- {a.iloc[i].open} --{a.iloc[i].rsi}--{a.iloc[i].close}")
#             print('='*20)
#             buy_price = a.iloc[i].close
#             p = []
#             check=1
#             continue
            
        if a.iloc[i-2].rsi<30 and a.iloc[i-1].rsi< 29.0 and  a.iloc[i].rsi >=34.0 :#and a.iloc[i].close>a.iloc[i].sma:
            print(f"{a.iloc[i].name}--- {a.iloc[i].open} -- {a.iloc[i].rsi}--{a.iloc[i].close}")
            print('='*20)
            rsi = a.iloc[i-1]
            lott = 3.0
            buy_price = a.iloc[i].close
            check=2
            continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 1.40
        p.append(a.iloc[i-1].rsi)
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}-- RSI {a.iloc[i].rsi} --{a.iloc[i].name}")
        if a.iloc[i].close > a.iloc[i].sma1 or a.iloc[i].rsi<=25.5:
            try:
                conti.append([pp, p[1], p[0], a.iloc[i].name])
            except:
                pass
            profit.append(pp)
#             print(f"RSI --- {abs(p[0])}")
            check=0
            print('-'*20)
#         if a.iloc[i-1].close>a.iloc[i-1].sma1 and a.iloc[i].close<a.iloc[i].sma1:
#             if a.iloc[i+1].close >= a.iloc[i+1].open:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_SELL) - 2.50
# #             elif a.iloc[i+1].close < a.iloc[i].sma2:
# #                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].low, mt5.ORDER_TYPE_BUY) - 2.50
#             else:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].high, mt5.ORDER_TYPE_SELL) - 2.50
#             print(f"TEST--  {pp1}  --")
# #             profit.append(pp1)
#             if pp1<-8:
#                 profit.append(-8)
#             else:
#                 profit.append(pp1)
            
    elif check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.5
        print(f"{pp}-- {a.iloc[i].close}-- RSI {a.iloc[i].rsi} --{a.iloc[i].name}")
        sub = a.iloc[i-1].rsi- a.iloc[i-2].rsi
        conti.append([pp, a.iloc[i-2].rsi, a.iloc[i-1].rsi, round(sub,2)])
        if a.iloc[i].rsi<a.iloc[i-1].rsi:
            profit.append(pp)
            check=0
#         else:
#             lott = lott - 1.0
#             if lott < 1.0:
#                 lott = 1.0
#             pp1 = price_action(symbol, lott, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) -2.5 
#             print(f"pp1 -- {pp1}-- {a.iloc[i].close}-- RSI {a.iloc[i].rsi} --{a.iloc[i].name}")
#             profit.append(pp1)
#         if a.iloc[i].close < a.iloc[i].sma2 or a.iloc[i].rsi>=78.5:
#             profit.append(pp)
#             check=0
#             print('-'*20)
#         if a.iloc[i-1].close<a.iloc[i-1].sma1 and a.iloc[i].close>a.iloc[i].sma1:
#             if a.iloc[i+1].close >= a.iloc[i+1].open:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - 1.4
# #             elif a.iloc[i+1].close < a.iloc[i].sma2:
# #                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].low, mt5.ORDER_TYPE_BUY) - 2.50
#             else:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].low, mt5.ORDER_TYPE_BUY) - 1.4
#             print(f"TEST--  {pp1}  --")
# #             profit.append(pp1)
#             if pp1<-8:
#                 profit.append(-8)
#             else:
#                 profit.append(pp1)

0.025
0.1
2024-04-23 11:15:00--- 66002.18 -- 38.20539409372451--66155.6
-2.36-- 66156.96-- RSI 38.339607896863164 --2024-04-23 11:30:00
3.99-- 66220.47-- RSI 44.86389001353687 --2024-04-23 11:45:00
11.74-- 66298.02-- RSI 52.08616271585256 --2024-04-23 12:00:00
5.02-- 66230.81-- RSI 45.99441705824584 --2024-04-23 12:15:00
2024-04-24 16:30:00--- 65998.8 -- 34.19019826800837--66140.31
-11.14-- 66053.89-- RSI 30.212864026818295 --2024-04-24 16:45:00
2024-04-24 19:30:00--- 64371.37 -- 37.1230049919067--64695.67
2.6399999999999997-- 64747.07-- RSI 39.436920131102156 --2024-04-24 19:45:00
20.17-- 64922.33-- RSI 47.170780426335824 --2024-04-24 20:00:00
20.82-- 64928.83-- RSI 47.461080300518596 --2024-04-24 20:15:00
20.7-- 64927.7-- RSI 47.408243271246555 --2024-04-24 20:30:00
2024-04-24 22:15:00--- 64190.11 -- 36.41495556346713--64342.52
-8.46-- 64282.9-- RSI 34.287916056819896 --2024-04-24 22:30:00
2024-04-24 23:30:00--- 63859.61 -- 42.54749850970748--64139.41
-12.79-- 64036.47-- RSI 38.36514

2024-06-18 05:15:00--- 64802.21 -- 40.9850607697258--65266.74
-0.2599999999999998-- 65289.12-- RSI 41.72000043221461 --2024-06-18 05:30:00
15.440000000000001-- 65446.12-- RSI 47.1106861636887 --2024-06-18 05:45:00
5.35-- 65345.28-- RSI 44.0570330877413 --2024-06-18 06:00:00
2024-06-18 18:30:00--- 64414.31 -- 47.867893498944866--64780.25
-11.48-- 64690.49-- RSI 44.36628282823405 --2024-06-18 18:45:00
2024-06-19 12:45:00--- 65033.71 -- 53.689159745353535--65252.0
-17.93-- 65097.68-- RSI 40.60148074004763 --2024-06-19 13:00:00
2024-06-19 22:45:00--- 64800.62 -- 35.4099578217122--64834.72
-9.620000000000001-- 64763.53-- RSI 28.71946682445804 --2024-06-19 23:00:00
2024-06-21 03:15:00--- 64680.18 -- 34.69543580834856--64774.27
4.2-- 64841.25-- RSI 44.30624827462064 --2024-06-21 03:30:00
-1.47-- 64784.58-- RSI 38.68634176688338 --2024-06-21 03:45:00
2024-06-21 07:15:00--- 64507.93 -- 34.54921263330999--64554.97
2.0600000000000005-- 64600.6-- RSI 40.842438586974225 --2024-06-21 07:30:00
5.03--

2024-08-02 18:30:00--- 62779.48 -- 40.428129457252666--63365.79
-5.84-- 63332.35-- RSI 39.91918683192953 --2024-08-02 18:45:00
2024-08-03 04:15:00--- 60742.47 -- 34.34717175324269--61112.52
-13.63-- 61001.2-- RSI 31.84809687232476 --2024-08-03 04:30:00
2024-08-03 19:45:00--- 60528.21 -- 34.927683860977936--60856.56
-11.43-- 60767.31-- RSI 32.71438247011116 --2024-08-03 20:00:00
2024-08-03 22:45:00--- 60053.44 -- 36.51369477779454--60197.13
18.51-- 60407.21-- RSI 49.309155259085614 --2024-08-03 23:00:00
31.46-- 60536.72-- RSI 55.72688379195509 --2024-08-03 23:15:00
26.4-- 60486.16-- RSI 52.68866159518231 --2024-08-03 23:30:00
2024-08-04 11:15:00--- 60199.3 -- 38.89585563459853--60380.52
13.649999999999999-- 60542.05-- RSI 51.378302807588405 --2024-08-04 11:30:00
11.67-- 60522.24-- RSI 49.919236346197856 --2024-08-04 11:45:00
2024-08-04 21:00:00--- 57806.95 -- 38.55222293205337--58165.72
12.69-- 58317.64-- RSI 42.97086753943817 --2024-08-04 21:15:00
22.43-- 58414.98-- RSI 45.880002336279

In [171]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

-380.85000000000014
Total negative sm -->-1132.88
Total negative -->93
Total positive sm -->752.0299999999999
Total positive -->36
Length 129


In [172]:
profit.sort()

In [173]:
profit

[-79.42,
 -52.27,
 -49.73,
 -46.08,
 -37.21,
 -28.94,
 -23.95,
 -21.73,
 -21.55,
 -20.36,
 -20.13,
 -19.38,
 -19.36,
 -18.92,
 -18.68,
 -17.93,
 -17.78,
 -17.14,
 -16.7,
 -15.73,
 -14.5,
 -14.23,
 -14.22,
 -14.19,
 -14.03,
 -13.7,
 -13.66,
 -13.63,
 -13.53,
 -13.5,
 -13.49,
 -12.86,
 -12.79,
 -12.75,
 -11.48,
 -11.44,
 -11.43,
 -11.14,
 -11.14,
 -11.12,
 -10.84,
 -10.42,
 -10.36,
 -9.89,
 -9.620000000000001,
 -9.379999999999999,
 -9.379999999999999,
 -9.27,
 -9.04,
 -8.46,
 -8.32,
 -8.23,
 -7.99,
 -7.95,
 -7.76,
 -7.74,
 -7.61,
 -7.53,
 -7.5,
 -6.8,
 -6.76,
 -6.58,
 -6.4399999999999995,
 -6.43,
 -6.42,
 -6.15,
 -6.12,
 -5.84,
 -5.720000000000001,
 -5.609999999999999,
 -5.41,
 -5.390000000000001,
 -5.27,
 -4.96,
 -4.83,
 -4.65,
 -4.16,
 -3.88,
 -3.86,
 -3.67,
 -3.44,
 -3.43,
 -3.35,
 -2.83,
 -2.82,
 -2.51,
 -2.04,
 -1.95,
 -1.9,
 -1.47,
 -1.27,
 -1.08,
 -0.73,
 1.8899999999999997,
 2.34,
 2.6100000000000003,
 3.1899999999999995,
 3.5300000000000002,
 4.45,
 4.63,
 4.99,
 5.02,
 5.3,
 5.

In [78]:
conti.sort()
conti

[[-98.79, 19.57828894269555, 46.56730711677777, 26.99],
 [-70.76, 28.54673798348078, 59.806088898364685, 31.26],
 [-31.97, 29.943843840607215, 40.50186201580658, 10.56],
 [-26.459999999999997, 17.498050573687806, 46.75032922767068, 29.25],
 [-25.45, 23.529532952883102, 65.58816970029716, 42.06],
 [-23.47, 25.52022160664896, 48.05524384980588, 22.54],
 [-22.52, 26.577340838817122, 40.932550591490376, 14.36],
 [-22.36, 29.87404162103023, 46.1816236458473, 16.31],
 [-21.32, 27.278896177336392, 44.698286565682515, 17.42],
 [-18.47, 20.5002525985404, 37.98286402684107, 17.48],
 [-18.11, 23.52515466324266, 46.74104320787265, 23.22],
 [-17.37, 17.759098631851458, 44.63361030828488, 26.87],
 [-15.48, 10.562190503934588, 36.37166675868178, 25.81],
 [-14.94, 28.047729955653068, 58.76395987128518, 30.72],
 [-14.66, 29.037076376672758, 39.52725886389597, 10.49],
 [-14.01, 18.45718798563132, 38.820291295539064, 20.36],
 [-12.9, 28.294061156035767, 36.46270372062765, 8.17],
 [-12.620000000000001, 28